# Pairwise Feature Regression 01 — directional OLS matrices (v1.0)

**Research question:** Which current-bar market features are individually linearly associated with other current-bar features, and how much of each feature remains unexplained by one other feature?

For every ordered pair of catalog features, fit **a separate simple OLS with an intercept**:
\[
Y_{i,t} = \alpha_{ij} + \beta_{ij} X_{j,t}+\varepsilon_{ij,t}.
\]
The **row is the response Y**, and the **column is the regressor X**. Results are computed **separately for each underlying** to avoid mixing assets with different units and market structures.

**Primary exports:** a directed matrix of raw slopes \(\beta_{ij}\) and a directed matrix of in-sample residual RMSE \(\sqrt{\sum_t \varepsilon_{ij,t}^{2}/n_{ij}}\). The slopes and errors are directional and generally asymmetric. Each pair uses only its own jointly available observations.

**Supporting exports:** n-by-pair, intercept, standardized slope (= Pearson correlation in one-predictor OLS), response-standardized RMSE, and \(R^2\). These diagnose units, unequal coverage and redundant information. A standardized slope or normalized RMSE alone is essentially a correlation matrix in disguise.

This is **descriptive**, *not* out-of-sample prediction, forecasting, feature causality, feature selection or a trading model. Time autocorrelation, session/clock effects and collector-version changes can create spurious associations. No significance tests are made and no future volatility labels are used.

### Feature universe
All **\`FEATURE_SPECS\`** predictors from \`full_information.py\`, with their original source-specific quote, microstructure and integrity masks, plus five-minute index return and the established clock/expiry conditioning features. Explicit raw audit/provenance columns, IDs, statuses and outcomes are excluded by design. An absent source still appears in the feature inventory and matrix as NULL; it is never silently imputed. This is the entire *model-ready numerical catalog*, not every numeric database column.

**Default sample:** all available five-minute feature states, *no cutoff*, one asset at a time. You may set time/asset filters in the config cell. Output folder: \`01/\`. This new branch does not change the HSMM or volatility-confirmation specifications.

## 1. Setup, scope and reproducibility

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from dotenv import load_dotenv

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Notebook must be launched from the repository or child directory")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from data_engineering.catalog.database import RESEARCH_DATABASE
from data_engineering.modeling.full_information import FEATURE_SPECS
from data_engineering.modeling.pairwise_feature_regression import (
    PAIRWISE_FEATURE_REGRESSION_VERSION, MIN_PAIR_OBSERVATIONS,
    EXTRAS, model_feature_frame, pairwise_ols, summarize_pairs
)

OUT = ROOT / "notebooks" / "Pairwise_Feature_Regression" / "01"
OUT.mkdir(parents=True, exist_ok=True)
MIN_PAIR_ROWS = MIN_PAIR_OBSERVATIONS
ASSETS = None  # None = every underlying; e.g. ["BTC", "ETH"]
START_DATE = None  # UTC lower bound, e.g. "2026-10-02"; None = no bound
END_DATE = None    # UTC upper bound inclusive, e.g. "2026-10-08"
MAKE_HEATMAPS = True

if not RESEARCH_DATABASE.is_file():
    raise FileNotFoundError(f"Local research DB unavailable: {RESEARCH_DATABASE}")
print("Version:", PAIRWISE_FEATURE_REGRESSION_VERSION)
print("DB:", RESEARCH_DATABASE)
print("Minimum jointly observed bars per ordered pair:", MIN_PAIR_ROWS)
subprocess.run([sys.executable, "-m", "unittest", "discover",
                "-s", str(ROOT / "tests"), "-p", "test_pairwise_feature_regression.py", "-v"],
               cwd=ROOT, check=True)
print("Synthetic tests passed.")

## 2. Read the existing engineered five-minute states — no writes to DuckDB

In [ ]:
def quote_id(name):
    return '"' + name.replace('"','""') + '"'

required_metadata = ["underlying", "state_time", "expiry_date"]
clock_sources = ["utc_minute_of_day", "session_progress_fraction",
                 "settlement_locked_fraction", "is_weekend", "minutes_to_expiry"]
readiness_sources = ["in_settlement_window", "perp_integrity_ok",
                     "microstructure_features_ready", "instability_features_ready",
                     "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
                     "option_snapshot_ready"]
source_columns = list(dict.fromkeys(
    required_metadata + clock_sources + readiness_sources +
    [spec.column for spec in FEATURE_SPECS] + [extra[1] for extra in EXTRAS]
))
with duckdb.connect(str(RESEARCH_DATABASE), read_only=True) as con:
    con.execute("SET TimeZone='UTC'")
    fields = set(con.execute(
        "DESCRIBE SELECT * FROM time_series_feature_state"
    ).df()["column_name"].tolist())
    absent = [c for c in source_columns if c not in fields]
    if not all(c in fields for c in required_metadata):
        raise ValueError(f"Required state keys missing: {set(required_metadata)-fields}")
    wanted = [c for c in source_columns if c in fields]
    query = "SELECT " + ", ".join(map(quote_id, wanted)) +             " FROM time_series_feature_state WHERE 1=1"
    params = []
    if START_DATE is not None:
        query += " AND state_time >= CAST(? AS TIMESTAMPTZ)"
        params.append(START_DATE + " 00:00:00+00")
    if END_DATE is not None:
        query += " AND state_time < CAST(? AS TIMESTAMPTZ)"
        params.append((pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1)).isoformat())
    if ASSETS is not None:
        if not ASSETS:
            raise ValueError("ASSETS must be None or a non-empty list")
        query += " AND underlying IN (" + ",".join(["?"] * len(ASSETS)) + ")"
        params.extend(ASSETS)
    query += " ORDER BY underlying, state_time"
    states = con.execute(query, params).df()
if states.empty:
    raise ValueError("No states selected; check date and underlying filters")
states["state_time"] = pd.to_datetime(states["state_time"], utc=True)
if states.duplicated(["underlying", "state_time"]).any():
    raise ValueError("Duplicate underlying/state_time found in engineered feature state")
print(f"Read {len(states):,} states across {states.underlying.nunique()} underlying assets")
print("Coverage:", states.state_time.min(), "to", states.state_time.max())
print("Missing catalog source columns:", absent)
display(states.groupby("underlying").agg(states=("state_time","size"),
                                          first=("state_time","min"),
                                          last=("state_time","max")))

## 3. Fit ALL ordered feature pairs, separately per asset

In [ ]:
results = {}
inventory_parts = []
summary = []
long_parts = []

for asset, part in states.groupby("underlying", sort=True):
    part = part.reset_index(drop=True)
    features, inventory = model_feature_frame(part)
    inventory.insert(0, "underlying", asset)
    inventory_parts.append(inventory)
    pair = pairwise_ols(features, min_n=MIN_PAIR_ROWS)
    results[asset] = (pair, inventory)

    # Stable, separately readable files for each underlying.
    matrices = {
        "beta_raw": pair.beta,
        "error_rmse": pair.rmse,
        "intercept": pair.intercept,
        "beta_standardized": pair.standardized_beta,
        "error_relative": pair.relative_rmse,
        "r_squared": pair.r_squared,
        "pair_counts": pair.n_pairs,
    }
    for name, matrix in matrices.items():
        matrix.to_csv(OUT / f"01_{asset}_{name}.csv", index_label="response")
    pairs = pair.long_form()
    pairs.insert(0, "underlying", asset)
    pairs.to_csv(OUT / f"01_{asset}_pairs_long.csv", index=False)
    long_parts.append(pairs)
    row = {"underlying": asset, "n_states": len(part),
           **summarize_pairs(pair)}
    row["n_source_present"] = int(inventory.source_present.sum())
    row["n_catalog_covered_ge_min"] = int((inventory.n_observed >= MIN_PAIR_ROWS).sum())
    summary.append(row)
    print(asset, row)

summary_df = pd.DataFrame(summary)
inventory_df = pd.concat(inventory_parts, ignore_index=True)
summary_df.to_csv(OUT / "01_summary.csv", index=False)
inventory_df.to_csv(OUT / "01_feature_inventory.csv", index=False)
display(summary_df)

## 4. Visualize beta and error matrices

The first heatmap shows **raw beta** with a signed logarithmic color scale to accommodate incomparable feature units. Its color scale is *not* an importance ranking across features. The second shows **raw residual RMSE** on a \(\log(1+\text{RMSE})\) display scale; CSVs hold the exact original values. The third shows **standardized beta**, which is directly comparable across pairs but equals the pairwise Pearson correlation.

The diagonal is blank: regressing a feature on itself is trivial. A blank off-diagonal cell means too few jointly observed rows, zero variance or invalid input. Consult \`01_{ASSET}_pair_counts.csv\` for the actual population behind every cell.

In [ ]:
def plot_matrix(matrix, asset, kind, output_path):
    z = matrix.to_numpy(float).copy()
    np.fill_diagonal(z, np.nan)
    finite = z[np.isfinite(z)]
    if not len(finite):
        print("Skip empty plot:", asset, kind)
        return
    if kind == "beta_raw":
        vmax = max(float(np.nanpercentile(np.abs(finite), 97)), 1e-8)
        lin = max(vmax / 100, 1e-8)
        values = np.clip(z, -vmax, vmax)
        cmap, norm, label = "coolwarm", SymLogNorm(
            linthresh=lin, vmin=-vmax, vmax=vmax
        ), "Raw slope β (signed-log color scale; clipped at 97th pct)"
    elif kind == "beta_standardized":
        values, cmap, norm, label = z, "coolwarm", None, "Standardized β = Pearson r"
    else:
        values, cmap, norm, label = np.log1p(np.maximum(z, 0)), "viridis", None,             "log(1 + in-sample raw RMSE)"
    fig, ax = plt.subplots(figsize=(17, 15))
    ax.set_facecolor("#eeeeee")
    if kind == "beta_standardized":
        im = ax.imshow(values, aspect="equal", interpolation="nearest",
                       cmap=cmap, vmin=-1, vmax=1)
    else:
        im = ax.imshow(values, aspect="equal", interpolation="nearest",
                       cmap=cmap, norm=norm)
    names = list(matrix.index)
    ax.set_xticks(np.arange(len(names)), labels=names, rotation=90, fontsize=5.5)
    ax.set_yticks(np.arange(len(names)), labels=names, fontsize=5.5)
    ax.set_xlabel("Predictor X (column)")
    ax.set_ylabel("Response Y (row)")
    ax.set_title(f"{asset} · {kind} · single-feature OLS (n ≥ {MIN_PAIR_ROWS})")
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.015)
    cb.set_label(label)
    fig.tight_layout()
    fig.savefig(output_path, dpi=125, bbox_inches="tight")
    plt.close(fig)

if MAKE_HEATMAPS:
    for asset, (pair, inventory) in results.items():
        # Omit features lacking sufficient observations only FROM PLOTS.
        # Full catalog stays present in every exported CSV matrix.
        visible = inventory.loc[
            inventory.n_observed >= MIN_PAIR_ROWS, "feature"
        ].tolist()
        if len(visible) < 2:
            print("No plottable features for", asset)
            continue
        for kind, matrix in (
            ("beta_raw", pair.beta),
            ("error_rmse", pair.rmse),
            ("beta_standardized", pair.standardized_beta),
        ):
            plot_matrix(matrix.loc[visible, visible], asset, kind,
                        OUT / f"01_{asset}_{kind}_heatmap.png")
    print("Heatmaps saved to", OUT)

## 5. Explore strongest standardized relationships without confusing them with prediction

In [ ]:
sample_asset = "BTC" if "BTC" in results else sorted(results)[0]
r, inv = results[sample_asset]
pairs = r.long_form()
pairs = pairs.loc[np.isfinite(pairs.standardized_beta)].copy()
pairs["abs_correlation"] = pairs.standardized_beta.abs()
# Undirected representatives for ranking correlation; directed raw beta still exported.
pairs = pairs.loc[pairs.response < pairs.predictor]
display(pairs.sort_values("abs_correlation", ascending=False)[
    ["response", "predictor", "n_pairs", "beta",
     "rmse", "standardized_beta", "relative_rmse"]
].head(25))
print("Example:", sample_asset,
      "| High correlation can reflect time-of-day, common trends, aliases,"
      " or sparse/selected observations, not directional predictability.")

## 6. Persist the reproducibility contract

In [ ]:
manifest = {
    "version": PAIRWISE_FEATURE_REGRESSION_VERSION,
    "study": "contemporaneous_directed_single_predictor_OLS",
    "source_relation": "time_series_feature_state",
    "feature_catalog": "full_information.FEATURE_SPECS + clock_core + EXTRAS",
    "n_catalog_specs": len(FEATURE_SPECS),
    "min_pair_observations": MIN_PAIR_ROWS,
    "as_of_execution_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start_date": START_DATE, "end_date": END_DATE, "assets_filter": ASSETS,
    "assets_covered": sorted(results),
    "n_input_rows": len(states),
    "first_state": str(states.state_time.min()),
    "last_state": str(states.state_time.max()),
    "missing_source_columns": absent,
    "matrix_axis": "row = response Y; column = predictor X",
    "beta": "unstandardized OLS slope, intercept fitted independently per pair",
    "rmse": "sqrt(mean in-sample residual**2) on EXACT pairwise complete rows",
    "no_imputation": True, "no_future_target": True,
    "no_global_complete_case": True, "no_hypothesis_tests": True,
    "database_read_only": True, "no_model_selection": True,
    "note": "Each off-diagonal cell may use different observations; check pair_counts",
}
(OUT / "01_run_metadata.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved", len(list(OUT.glob("01_*"))), "artifacts in:", OUT)
print("Commit 01/ CSV, JSON, PNG outputs and the executed notebook for review.")

## Interpretive limits and v2 candidates (not included here)

1. **Scale:** raw slopes and raw RMSE are in transformed response/predictor units; they cannot be directly ranked across arbitrary features. Standardized slope equals correlation and is symmetric on paired samples; it is a check, not a separate regression model.
2. **Coverage:** as-of-now raw L2 vintage, option quote restrictions, 0DTE settlement, and market data gaps may change pairwise eligibility. \`pair_counts\` and the inventory must be read before interpreting high slopes/low errors.
3. **Timing:** all values describe the same completed five-minute bar. No temporal leads or lags are tested. No statements about forecasting or trading are justified.
4. **Dependence:** independent regressions do **not** condition on all other features; relationships may arise from market-wide movement, clock cycle, shared denominators, or redundant constructions. No statistical significance is reported.
5. **Future work only with approval:** by-expiry-window/clock and by-collector-vintage matrices; out-of-sample directional tests; residualizing common clock effects; uncertainty via session block bootstrap. None of these change this v1 specification.